In [1]:
print("here")

here


In [2]:
import logging
import os
from pathlib import Path
from dotenv import load_dotenv
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

# Credentials are loaded from .env at the repo root (not committed to git).
# Copy .env.example → .env at the repo root and fill in your values.
_repo_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
load_dotenv(_repo_root / ".env")

SERVER_ENDPOINT          = os.environ.get("SERVER_ENDPOINT", "")
DATABRICKS_WORKSPACE_URL = os.environ.get("DATABRICKS_WORKSPACE_URL", "")

from databricks.sdk import WorkspaceClient
_w = WorkspaceClient()

_url_mismatch = DATABRICKS_WORKSPACE_URL and DATABRICKS_WORKSPACE_URL != _w.config.host
if _url_mismatch:
    print(f"Warning: DATABRICKS_WORKSPACE_URL={DATABRICKS_WORKSPACE_URL!r} does not match "
          f"WorkspaceClient host={_w.config.host!r}. Recalculating both values.")
    SERVER_ENDPOINT = ""
    DATABRICKS_WORKSPACE_URL = ""

if not SERVER_ENDPOINT or not DATABRICKS_WORKSPACE_URL:
    if not DATABRICKS_WORKSPACE_URL:
        DATABRICKS_WORKSPACE_URL = _w.config.host
    if not SERVER_ENDPOINT:
        _workspace_id = _w.get_workspace_id()
        if "azuredatabricks.net" in DATABRICKS_WORKSPACE_URL:
            _region = _w.clusters.list_zones().default_zone          # Azure: zone == region
        else:
            _region = _w.clusters.list_zones().default_zone[:-1]     # AWS: strip trailing AZ letter
        _domain = "azuredatabricks.net" if "azuredatabricks.net" in DATABRICKS_WORKSPACE_URL else "cloud.databricks.com"
        SERVER_ENDPOINT = f"https://{_workspace_id}.zerobus.{_region}.{_domain}"
    print(f"Auto-derived SERVER_ENDPOINT={SERVER_ENDPOINT}")
    print(f"Auto-derived DATABRICKS_WORKSPACE_URL={DATABRICKS_WORKSPACE_URL}")
TABLE_NAME    = os.environ["TABLE_NAME"]
CLIENT_ID     = os.environ.get("CLIENT_ID", "")
CLIENT_SECRET = os.environ.get("CLIENT_SECRET", "")

def _credentials_valid(client_id: str, client_secret: str) -> bool:
    if not client_id or not client_secret:
        return False
    try:
        from databricks.sdk import WorkspaceClient as _WC
        _test = _WC(host=DATABRICKS_WORKSPACE_URL, client_id=client_id, client_secret=client_secret)
        _test.current_user.me()
        return True
    except Exception:
        return False

if not _credentials_valid(CLIENT_ID, CLIENT_SECRET):
    import time
    print("CLIENT_ID/CLIENT_SECRET missing or invalid — creating a new service principal (session only, not saved to .env)")
    _sp = _w.service_principals.create(display_name=f"zerobusdemo-{int(time.time())}")
    _secret = _w.service_principal_secrets.create(service_principal_id=_sp.id)
    CLIENT_ID     = str(_sp.application_id)
    CLIENT_SECRET = _secret.secret
    print(f"Created service principal CLIENT_ID={CLIENT_ID} (not persisted)")

In [3]:
# Manual cleanup cell — run this only if you need to reset the table between full runs.
# NOT part of Run All.
# spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")

In [4]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {'.'.join(TABLE_NAME.split('.')[:2])}")

# Drop and recreate if the schema has been corrupted by a previous Scenario 2 run.
_expected = {"device_name", "temp", "humidity"}
try:
    _actual = {f.name for f in spark.table(TABLE_NAME).schema}
    if not _expected.issubset(_actual):
        print(f"Schema mismatch (found {_actual}), dropping and recreating...")
        spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
except Exception:
    pass  # table doesn't exist yet — CREATE below will handle it

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
  device_name STRING,
  temp        INT,
  humidity    INT
)
""")
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {TABLE_NAME} TO `{CLIENT_ID}`")

Schema mismatch (found {'device_name', 'sensor_id'}), dropping and recreating...


""


In [5]:
_catalog, _schema, _table = TABLE_NAME.split(".")
_table_url = f"{DATABRICKS_WORKSPACE_URL.rstrip('/')}/explore/data/{_catalog}/{_schema}/{_table}"
print(f"Table: {_table_url}")

Table: https://e2-demo-field-eng.cloud.databricks.com/explore/data/main/robert_lee/air_quality


In [6]:
_detail = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").collect()[0]
_location = _detail["location"]
_rejected_path = f"{_location}/_zerobus/table_rejected_parquets/"

print(f"Table storage : {_location}")
print(f"Rejected rows : {_rejected_path}")

Table storage : s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/0cde10aa-8192-4037-a0bd-07e5da1a4d05
Rejected rows : s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/0cde10aa-8192-4037-a0bd-07e5da1a4d05/_zerobus/table_rejected_parquets/


In [7]:
# Normal ingest — 10 rows of valid data matching the table schema
sdk = ZerobusSdk(
    SERVER_ENDPOINT,
    DATABRICKS_WORKSPACE_URL
)

table_properties = TableProperties(TABLE_NAME)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)

try:
    last_offset = None
    for i in range(10):
        record_dict = {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        last_offset = stream.ingest_record_offset(record_dict)

    # Wait once for the final offset — all prior offsets are guaranteed committed too
    if last_offset is not None:
        stream.wait_for_offset(last_offset)
finally:
    stream.close()


2026-03-24T22:01:54.742136Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=46692e8d-160a-4dac-bf46-490a025a9e40
2026-03-24T22:01:54.742186Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=46692e8d-160a-4dac-bf46-490a025a9e40
2026-03-24T22:01:54.742198Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=46692e8d-160a-4dac-bf46-490a025a9e40
2026-03-24T22:01:54.743034Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=46692e8d-160a-4dac-bf46-490a025a9e40
2026-03-24T22:01:54.945226Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Waiting for acknowledgement completed. stream_id=46692e8d-160a-4dac-bf46-490a025a9e40
2026-03-24T22:01:54.945269Z  INFO databricks_zerobus_ingest_sdk: Closing stream stream_id=46692e8d-160a-4dac-bf46-490a025a9e40
2026-03-24T22:01:54.945274Z  INFO databricks_zerobus_ingest_sd

In [8]:
spark.sql(f"select min(device_name), max(device_name), count(*) from {TABLE_NAME}")


,min(device_name),max(device_name),count(1)
0,NaN,NaN,0


In [9]:
spark.sql(f"""
SELECT device_name, count(*) AS row_count
FROM {TABLE_NAME}
GROUP BY device_name
HAVING count(*) > 1
ORDER BY CAST(regexp_extract(device_name, '(\\\\d+)$', 1) AS INT), row_count DESC
""")

,device_name,row_count


In [10]:
# Scenario 1 — schema mismatch caught BEFORE acknowledgement
# ZeroBus detects the unknown field at the protocol level and closes the stream immediately.
# Result: ZerobusException raised, no data committed, nothing written to _zerobus/table_rejected_parquets/.
# Trigger: send a field name that does not exist in the table schema ("device_name_2" vs "device_name").

sdk = ZerobusSdk(
    SERVER_ENDPOINT,
    DATABRICKS_WORKSPACE_URL
)

table_properties = TableProperties(TABLE_NAME)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)

try:
    last_offset = None
    for i in range(10):
        record_dict = {
            "device_name_2": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        last_offset = stream.ingest_record_offset(record_dict)

    if last_offset is not None:
        stream.wait_for_offset(last_offset)
except Exception as e:
    print(f"Expected ZerobusException (Scenario 1): {e}")
finally:
    stream.close()

2026-03-24T22:01:58.910152Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=75aa1127-8a89-40c9-b1ab-bd1743682f78
2026-03-24T22:01:58.910177Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=75aa1127-8a89-40c9-b1ab-bd1743682f78
2026-03-24T22:01:58.910202Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=75aa1127-8a89-40c9-b1ab-bd1743682f78
2026-03-24T22:01:58.911850Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=75aa1127-8a89-40c9-b1ab-bd1743682f78
2026-03-24T22:01:58.987389Z ERROR databricks_zerobus_ingest_sdk: Unexpected response from server Status { code: InvalidArgument, message: "Record decoder/encoder error: unrecognized field name 'device_name_2' at line 1 column 16. Error Code: 4044, Error State: 3.", source: None }
2026-03-24T22:01:58.987412Z ERROR databricks_zerobus_ingest_sdk: Stream failure detected: Stream is 

In [11]:
# Scenario 2 — post-acknowledgement rejection (durable fallback)
#
# ENVIRONMENT REQUIREMENT: this cell must run on a CLASSIC CLUSTER inside the
# Databricks workspace (not via Databricks Connect). The schema-change thread
# uses spark.write.save(_location) which writes directly to the S3 Delta log.
# From Databricks Connect that call is IAM-blocked on managed buckets.
# On a classic cluster in the workspace, the cluster's UC credential passthrough
# provides the necessary S3 access to the managed path.
#
# Why saveAsTable / CREATE OR REPLACE TABLE do NOT work:
#   Both create a new managed storage path and swap the UC pointer atomically.
#   ZeroBus holds the original path from stream-creation time and writes there
#   unaffected. Only a direct write to the original _delta_log path triggers fallback.
#
# Timing pattern:
#   - Send 300 rows slowly (0.01s/row ≈ 3s) with wait_for_offset every 30 rows
#   - Schema-change thread fires at 2s and overwrites the Delta log at _location
#   - ZeroBus's next commit after the overwrite sees an incompatible schema → fallback

import threading
import time
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

sdk2 = ZerobusSdk(SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL)
stream2 = sdk2.create_stream(
    CLIENT_ID, CLIENT_SECRET,
    TableProperties(TABLE_NAME),
    StreamConfigurationOptions(record_type=RecordType.JSON),
)
print("Stream opened.")

# --- schema-change thread: fires at 2s, writes directly to _location via S3 ---
def change_schema():
    time.sleep(2)
    print("[schema-thread] Overwriting Delta log at original path (removing temp + humidity)...")
    incompatible = StructType([
        StructField("device_name", StringType(), True),
        StructField("sensor_id",   IntegerType(), True),
    ])
    (spark.createDataFrame([], incompatible)
         .write.format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .save(_location))    # direct S3 path — requires on-cluster UC credentials
    print("[schema-thread] Schema overwrite complete.")

schema_thread = threading.Thread(target=change_schema, daemon=True)
schema_thread.start()

# --- send 300 rows slowly with a wait_for_offset commit every 30 rows ---
last_offset = None
try:
    for i in range(300):
        last_offset = stream2.ingest_record_offset({
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40,
        })
        time.sleep(0.01)
        if (i + 1) % 30 == 0:
            stream2.wait_for_offset(last_offset)
            print(f"Committed rows 0–{i}...")

    if last_offset is not None:
        stream2.wait_for_offset(last_offset)
    print("All 300 rows committed — schema change may not have landed in time.")
except Exception as e:
    print(f"Stream error (expected — schema mismatch): {e}")
finally:
    stream2.close()

schema_thread.join()

# --- restore original schema (always runs) ---
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_NAME} (
      device_name STRING,
      temp        INT,
      humidity    INT
    )
    """)
    spark.sql(f"GRANT MODIFY, SELECT ON TABLE {TABLE_NAME} TO `{CLIENT_ID}`")
    print("Table restored — check the next cell for rejected rows.")
except Exception as e:
    print(f"Restore failed (fix manually): {e}")

2026-03-24T22:02:00.076341Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=528c8a09-29be-4435-b616-466d0045ba37
2026-03-24T22:02:00.076362Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=528c8a09-29be-4435-b616-466d0045ba37
Stream opened.
2026-03-24T22:02:00.076391Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=528c8a09-29be-4435-b616-466d0045ba37
2026-03-24T22:02:00.410548Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to offset 19. Waiting for offset 29. stream_id=528c8a09-29be-4435-b616-466d0045ba37
Committed rows 0–29...
2026-03-24T22:02:00.578541Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Waiting for acknowledgement completed. stream_id=528c8a09-29be-4435-b616-466d0045ba37
2026-03-24T22:02:00.905805Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to offset 41. Waiting for offset 59. stream_id=528c8a09-29be-4435-b616-466d0045ba3

In [12]:
# Check rejected rows written by Scenario 2
print(f"Rejected rows path:\n  {_rejected_path}\n")
try:
    _rejected_df = spark.sql(f"SELECT * FROM parquet.`{_rejected_path}`")
    _count = _rejected_df.count()
    if _count > 0:
        print(f"Found {_count} rejected row(s):")
        display(_rejected_df)
    else:
        print("Path exists but no rejected rows found.")
except Exception as e:
    _msg = str(e)
    if any(x in _msg for x in ["Path does not exist", "FileNotFoundException", "NoSuchKey", "NoSuchBucket", "is not a Parquet file"]):
        print("No rejected rows found — Scenario 2 did not produce fallback data.")
        print("Scenario 2 requires the table schema to change between ZeroBus's")
        print("internal acknowledgement and its Delta commit. ZeroBus materializes")
        print("too quickly for a DROP TABLE from Python to create this window.")
    elif "DELTA_INVALID_FORMAT" in _msg:
        print("Running on serverless — cannot read plain Parquet inside a Delta root.")
        print("Switch to a classic cluster, then run:")
        print(f"  SELECT * FROM parquet.`{_rejected_path}`")
    else:
        raise

Rejected rows path:
  s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/0cde10aa-8192-4037-a0bd-07e5da1a4d05/_zerobus/table_rejected_parquets/

Running on serverless — cannot read plain Parquet inside a Delta root.
Switch to a classic cluster, then run:
  SELECT * FROM parquet.`s3://databricks-e2demofieldengwest/b169b504-4c54-49f2-bc3a-adf4b128f36d/tables/0cde10aa-8192-4037-a0bd-07e5da1a4d05/_zerobus/table_rejected_parquets/`


In [13]:
# ZeroBus system table — primary way to verify rejections from Databricks Connect.
# Rows where errors is non-null are commits that failed and whose data was written
# to _zerobus/table_rejected_parquets/ instead of the Delta table.
spark.sql(f"""
SELECT commit_time, table_name, committed_records, errors
FROM system.lakeflow.zerobus_ingest
WHERE table_name = '{TABLE_NAME}'
ORDER BY commit_time DESC
LIMIT 20
""")

,commit_time,table_name,committed_records,errors
0,2026-03-24 21:36:41.430,main.robert_lee.air_quality,20,[]
1,2026-03-24 21:36:36.414,main.robert_lee.air_quality,10,[]
2,2026-03-24 21:32:12.939,main.robert_lee.air_quality,20,[]
3,2026-03-24 21:32:07.792,main.robert_lee.air_quality,10,[]
4,2026-03-24 21:23:32.966,main.robert_lee.air_quality,200,[]
5,2026-03-24 21:23:28.017,main.robert_lee.air_quality,10,[]
6,2026-03-24 21:16:07.334,main.robert_lee.air_quality,200,[]
7,2026-03-24 21:16:02.373,main.robert_lee.air_quality,10,[]
8,2026-03-24 21:11:16.413,main.robert_lee.air_quality,1,[]
9,2026-03-24 21:11:11.378,main.robert_lee.air_quality,21,[]
